In [ ]:
import optuna
import numpy as np
import pandas as pd
import yfinance as yf
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

import gymnasium as gym
from gymnasium import spaces

import gc

# Environnement Forex compatible Gymnasium

In [2]:
class ForexEnv(gym.Env):
    def __init__(self, df, initial_balance=1000):
        super(ForexEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        self.initial_balance = float(initial_balance)
        self.balance = float(initial_balance)
        self.usd = 0.0
        self.current_step = 0

        # Observation: [prix, balance, usd]
        self.observation_space = spaces.Box(
            low=0, high=np.inf, shape=(3,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(3)  # 0=Hold, 1=Buy, 2=Sell

    def _get_obs(self):
        price = float(self.df.loc[self.current_step, "Close"])
        return np.array([price, self.balance, self.usd], dtype=np.float32)

    def step(self, action):
        price = float(self.df.loc[self.current_step, "Close"])
        reward = 0.0

        # BUY USD
        if action == 1 and self.balance > 0:
            self.usd += self.balance / price
            self.balance = 0.0
        # SELL USD
        elif action == 2 and self.usd > 0:
            self.balance += self.usd * price
            self.usd = 0.0
            reward = self.balance - self.initial_balance

        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1
        truncated = False
        obs = self._get_obs()
        info = {}
        return obs, float(reward), terminated, truncated, info

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.balance = float(self.initial_balance)
        self.usd = 0.0
        self.current_step = 0
        obs = self._get_obs()
        info = {}
        return obs, info

# Chargement des données EUR/USD

In [3]:
df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d")
df = df.dropna().reset_index(drop=True)

C:\Users\Aurelien\AppData\Local\Temp\ipykernel_54060\1899308409.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d")
[*********************100%***********************]  1 of 1 completed


# Fonction d'optimisation Optuna

In [ ]:
def optimize_ppo(trial):
    # === Hyperparamètres à tester ===
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-3)
    n_steps = trial.suggest_categorical("n_steps", [1024, 2048, 4096])
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    gamma = trial.suggest_uniform("gamma", 0.90, 0.999)
    ent_coef = trial.suggest_uniform("ent_coef", 0.0, 0.02)

    # === Création de l'environnement ===
    env = DummyVecEnv([lambda: Monitor(ForexEnv(df))])

    # === Modèle PPO ===
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        gamma=gamma,
        ent_coef=ent_coef,
        tensorboard_log="./logs/optuna_forex",
        verbose=0,
    )

    # === Entraînement court pour test ===
    model.learn(total_timesteps=100_000)

    # === Évaluation sur 5 épisodes ===
    mean_reward, _ = evaluate_policy(model, env, n_eval_episodes=5, deterministic=True)

    # === Nettoyage mémoire / éviter deepcopy crash ===
    model.env.close()
    env.close()

    # On ne stocke pas directement le modèle (non picklable)
    trial.set_user_attr("mean_reward", float(mean_reward))

    # Forcer la libération mémoire
    del model, env
    gc.collect()

    # === Apprentissage par l’échec ===
    # (on pénalise les mauvaises récompenses)
    if mean_reward < 0:
        return -1000 + mean_reward
    return mean_reward


# Lancer la recherche Optuna

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(optimize_ppo, n_trials=2)

# ✅ Contourne l’erreur deepcopy du best_trial
best_trial = max(study.trials, key=lambda t: t.value if t.value is not None else float('-inf'))

print("✅ Best trial:")
print("  Reward:", best_trial.value)
print("  Params:", best_trial.params)

# === Optionnel : réentraîner le meilleur modèle proprement ===
best_params = best_trial.params
env = DummyVecEnv([lambda: Monitor(ForexEnv(df))])
best_model = PPO("MlpPolicy", env, **best_params, verbose=0)
best_model.learn(total_timesteps=500_000)
best_model.save("models/best_ppo_forex.zip")
env.close()

[I 2025-10-19 12:05:42,072] A new study created in memory with name: no-name-fbebc45e-0a8f-4d4d-b52a-1d77ac309c9a
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_54060\2131879411.py:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-3)
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_54060\2131879411.py:6: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  gamma = trial.suggest_uniform("gamma", 0.90, 0.999)
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_54060\2131879411.py:7: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0

✅ Best trial:


TypeError: cannot pickle '_thread.lock' object